# Stage 3 · Dream-v0-Instruct-7B: answer slots and scoring-code tests

Runtime: **A100** (Dream needs at least 20 GB). One fresh runtime for this notebook only. Dream's logits are shifted right by one position; test 2 is also the shift check.

In [ ]:
BRANCH = "main"   # the branch to test; main once the stage-3 PR is merged
!nvidia-smi --query-gpu=name,memory.total --format=csv
!git clone -q -b {BRANCH} https://github.com/Manas-33/CSCI-544-Course-Project.git /content/project
%cd /content/project
!git log --oneline -1
!pip install -q -r harness/diffusion/requirements/dream.txt
OUT = "results/stage3/dream"
!mkdir -p {OUT}
import hashlib
DEV_HASH = hashlib.sha256(open("bank/dev/dev_bank.jsonl", "rb").read()).hexdigest()
SANITY_HASH = hashlib.sha256(open("bank/sanity/sanity_set.jsonl", "rb").read()).hexdigest()
print(DEV_HASH, SANITY_HASH)

**After the install cell finishes:** Dream pins torch 2.5.1, which replaces Colab's torch. Open *Runtime → Restart session*, then run the next cell, which goes back into the project folder.

In [ ]:
%cd /content/project
OUT = "results/stage3/dream"
!mkdir -p {OUT}
import hashlib
DEV_HASH = hashlib.sha256(open("bank/dev/dev_bank.jsonl", "rb").read()).hexdigest()
SANITY_HASH = hashlib.sha256(open("bank/sanity/sanity_set.jsonl", "rb").read()).hexdigest()
print(DEV_HASH, SANITY_HASH)
import torch, transformers
print(torch.__version__, transformers.__version__)   # expect 2.5.1 and 4.46.2

## 1. Choose the answer-slot count on the development bank (PLAN.md §4)
Scores the 142 development answers with 1, 8 and 32 slots (labels are never read), then picks the highest median Yes+No mass, ties to fewer slots. Also reports the usable-scores gate.

In [ ]:
for n in (1, 8, 32):
    !python -m harness.diffusion.run_score --model dream --n-slots {n} --items bank/dev/dev_bank.jsonl --bank-hash {DEV_HASH} --out {OUT}/dev_T1_n{n}.jsonl
!python -m harness.diffusion.choose_slots {OUT}/dev_T1_n1.jsonl {OUT}/dev_T1_n8.jsonl {OUT}/dev_T1_n32.jsonl --report {OUT}/slots.json
import json
choice = json.load(open(f"{OUT}/slots.json"))
N, PREFIX = choice["chosen_n_slots"], choice["chosen_answer_prefix"]
print("answer slots:", N, "| answer prefix:", PREFIX)

## 2. Smoke test on a tiny random-weight copy of the official code (CPU)

In [ ]:
!python -m harness.diffusion.checks.check_tiny --model dream --answer-prefix {PREFIX} --report {OUT}/tiny.json

## 3. Scoring-code tests 1–3 (PLAN.md §4)
Test 2 compares our scorer with the official model code's first forward pass; test 1 is the sanity bar (AUROC ≥ 0.90); test 3 checks batched against unbatched scores and repeat runs.

In [ ]:
!python -m harness.diffusion.checks.check_parity --model dream --n-slots {N} --answer-prefix {PREFIX} --report {OUT}/test2_parity_n{N}.json
!python -m harness.diffusion.checks.check_sanity --model dream --n-slots {N} --answer-prefix {PREFIX} --report {OUT}/test1_sanity_n{N}.json --out {OUT}/sanity_scores_n{N}.jsonl
!python -m harness.diffusion.checks.check_invariance --model dream --n-slots {N} --answer-prefix {PREFIX} --report {OUT}/test3_invariance_n{N}.json --save {OUT}/invariance_scores_n{N}.json

Each check prints PASS or FAIL and writes its report even when it fails; a FAIL is a result, not a crash. Keep going.

## Download the results
This zips `results/stage3/dream/` (scores, manifests and reports). Send the zip back, or unzip it into the repo at the same path.

In [ ]:
!cd /content/project && zip -qr /content/stage3_dream.zip results/stage3/dream
from google.colab import files
files.download("/content/stage3_dream.zip")